# IBD scRNA-seq Pipeline — Master Notebook
**Dataset:** GSE134809 (Smillie et al.) — ileal biopsies, Involved vs Uninvolved  
**Project:** Ritschel Research / ibd-scrna  
**Scripts:** 01 Load & QC → 02 scVI → 03 Annotate → 04 Signatures → 05 DE → 06 LINCS → 07 Novelty  

Run cells top-to-bottom. Each script section saves its output to Drive before proceeding.

---
## 0. Setup: Mount Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%capture
!pip install scanpy anndata scvi-tools gseapy

In [ ]:
# ── Shared paths (used by all scripts) ────────────────────────────────────
import os

DRIVE_BASE   = "/content/drive/MyDrive/Ritschel_Research/ibd_scrna_output"
RAW_DIR      = os.path.join(DRIVE_BASE, "raw")
PROCESSED    = os.path.join(DRIVE_BASE, "processed")

for d in [DRIVE_BASE, RAW_DIR, PROCESSED]:
    os.makedirs(d, exist_ok=True)

print("Drive base: ", DRIVE_BASE)
print("Raw dir:    ", RAW_DIR)
print("Processed:  ", PROCESSED)
print("Files in raw:", len(os.listdir(RAW_DIR)))

---
## Script 01 — Load & QC
Flat MTX layout. Discovers all GSM samples, renames `genes.tsv.gz` → `features.tsv.gz` per sample, runs QC filters, normalises, selects HVGs.  
**Output:** `processed/01_loaded.h5ad`

In [ ]:
import gc, re, shutil, tempfile
import numpy as np
import scanpy as sc

sc.settings.verbosity = 1

# ── Condition map ─────────────────────────────────────────────────────────
# GSE134809 ileal biopsies: odd GSM = Involved, even = Uninvolved (original 22)
# GSM4761136-144: 9 additional — verify against GEO page if results look odd
CONDITION_MAP = {
    # Involved (active Crohn's)
    "GSM3972009": ("Involved",    "69"),
    "GSM3972011": ("Involved",   "122"),
    "GSM3972013": ("Involved",   "128"),
    "GSM3972016": ("Involved",   "138"),
    "GSM3972017": ("Involved",   "158"),
    "GSM3972020": ("Involved",   "181"),
    "GSM3972022": ("Involved",   "187"),
    "GSM3972024": ("Involved",   "190"),
    "GSM3972026": ("Involved",   "193"),
    "GSM3972028": ("Involved",   "196"),
    "GSM3972030": ("Involved",   "209"),
    # Uninvolved (paired non-inflamed)
    "GSM3972010": ("Uninvolved",  "68"),
    "GSM3972012": ("Uninvolved", "123"),
    "GSM3972014": ("Uninvolved", "129"),
    "GSM3972015": ("Uninvolved", "135"),
    "GSM3972018": ("Uninvolved", "159"),
    "GSM3972019": ("Uninvolved", "180"),
    "GSM3972021": ("Uninvolved", "186"),
    "GSM3972023": ("Uninvolved", "189"),
    "GSM3972025": ("Uninvolved", "192"),
    "GSM3972027": ("Uninvolved", "195"),
    "GSM3972029": ("Uninvolved", "208"),
    # Additional samples — verify conditions from GEO
    "GSM4761136": ("Uninvolved",  "67"),
    "GSM4761137": ("Involved",   "126"),
    "GSM4761138": ("Uninvolved", "127"),
    "GSM4761139": ("Uninvolved", "134"),
    "GSM4761140": ("Uninvolved", "157"),
    "GSM4761141": ("Uninvolved", "179"),
    "GSM4761142": ("Uninvolved", "185"),
    "GSM4761143": ("Uninvolved", "191"),
    "GSM4761144": ("Uninvolved", "194"),
}

involved_n   = sum(1 for v in CONDITION_MAP.values() if v[0] == "Involved")
uninvolved_n = sum(1 for v in CONDITION_MAP.values() if v[0] == "Uninvolved")
print(f"Condition map: {involved_n} Involved, {uninvolved_n} Uninvolved")

In [ ]:
def discover_samples(raw_dir):
    """Scan flat directory and group files by GSM ID."""
    samples = {}
    for fname in os.listdir(raw_dir):
        m = re.match(r'^(GSM\d+)_\d+_(barcodes|genes|matrix)\.(tsv|mtx)\.gz$', fname)
        if not m:
            continue
        gsm, ftype = m.group(1), m.group(2)
        if gsm not in samples:
            samples[gsm] = {}
        samples[gsm][ftype] = os.path.join(raw_dir, fname)
    return samples

def load_sample(gsm, paths, condition, patient):
    """Copy flat files to a temp dir with standard names; rename genes→features for scanpy."""
    tmp = tempfile.mkdtemp()
    try:
        shutil.copy(paths["barcodes"], os.path.join(tmp, "barcodes.tsv.gz"))
        shutil.copy(paths["matrix"],   os.path.join(tmp, "matrix.mtx.gz"))
        shutil.copy(paths["genes"],    os.path.join(tmp, "features.tsv.gz"))  # CellRanger v2 rename
        adata = sc.read_10x_mtx(tmp, var_names="gene_symbols", cache=False)
        adata.obs["condition"] = condition
        adata.obs["patient"]   = patient
        adata.obs["sample"]    = gsm
        adata.obs_names        = [f"{gsm}_{bc}" for bc in adata.obs_names]
        return adata
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

print("Discovering samples...")
all_samples = discover_samples(RAW_DIR)
print(f"Found {len(all_samples)} GSM IDs")

unknown = [g for g in all_samples if g not in CONDITION_MAP]
if unknown:
    print(f"WARNING — no condition mapping for: {unknown} (will be skipped)")

adatas_all = []
for cond in ["Involved", "Uninvolved"]:
    print(f"\nLoading {cond} samples...")
    adatas = []
    for gsm in sorted(all_samples):
        if gsm not in CONDITION_MAP:
            continue
        c, patient = CONDITION_MAP[gsm]
        if c != cond:
            continue
        paths = all_samples[gsm]
        if not all(k in paths for k in ["barcodes", "genes", "matrix"]):
            print(f"  {gsm} — SKIP (missing: {set(['barcodes','genes','matrix']) - set(paths)})")
            continue
        try:
            adata = load_sample(gsm, paths, cond, patient)
            print(f"  {gsm} (patient {patient}) — {adata.n_obs} cells")
            adatas.append(adata)
        except Exception as e:
            print(f"  {gsm} — ERROR: {e}")
    if not adatas:
        print(f"  WARNING: no {cond} samples loaded")
        continue
    print(f"  Concatenating {len(adatas)} {cond} samples...")
    batch = sc.concat(adatas, join="outer", fill_value=0)
    del adatas; gc.collect()
    print(f"  {cond}: {batch.n_obs:,} cells")
    adatas_all.append(batch)

print("\nMerging conditions...")
adata = sc.concat(adatas_all, join="outer", fill_value=0)
del adatas_all; gc.collect()
print(f"Combined: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

In [ ]:
# QC metrics
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)

print("Pre-filter summary:")
print(f"  Cells:              {adata.n_obs:,}")
print(f"  Genes:              {adata.n_vars:,}")
print(f"  Median genes/cell:  {adata.obs['n_genes_by_counts'].median():.0f}")
print(f"  Median MT%:         {adata.obs['pct_counts_mt'].median():.1f}%")

# Inspect distributions before committing to thresholds
sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
             jitter=0.4, multi_panel=True)

In [ ]:
# ── Adjust thresholds here based on violin plots above ────────────────────
MIN_GENES  = 200
MAX_GENES  = 6000
MAX_MT_PCT = 20

n_before = adata.n_obs
adata = adata[adata.obs["n_genes_by_counts"] >= MIN_GENES].copy()
adata = adata[adata.obs["n_genes_by_counts"] <= MAX_GENES].copy()
adata = adata[adata.obs["pct_counts_mt"]     <= MAX_MT_PCT].copy()
print(f"After QC: {adata.n_obs:,} cells ({n_before - adata.n_obs:,} removed)")
print(f"  Thresholds: genes {MIN_GENES}–{MAX_GENES}, MT ≤{MAX_MT_PCT}%")

In [ ]:
# Normalise + HVG
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata.copy()

sc.pp.highly_variable_genes(adata, n_top_genes=3000, batch_key="sample", subset=True)
print(f"HVGs retained: {adata.n_vars}")

out_01 = os.path.join(PROCESSED, "01_loaded.h5ad")
adata.write_h5ad(out_01)
print(f"Saved: {out_01}")
print("\nCondition breakdown:")
print(adata.obs["condition"].value_counts().to_string())
print("\nScript 01 complete ✓")

---
## Script 02 — scVI Embedding & Leiden Clustering
scVI (n_latent=30, n_layers=2, n_hidden=128, 400 epochs). Multi-resolution Leiden (0.5 / 0.8 / 1.2).  
**Requires T4 GPU.**  
**Output:** `processed/02_scvi.h5ad`

In [ ]:
import scvi
import torch
import random
import pandas as pd

SCVI_PARAMS        = {"n_latent": 30, "n_layers": 2, "n_hidden": 128}
SCVI_TRAIN_PARAMS  = {"max_epochs": 400, "early_stopping": False}
N_NEIGHBORS        = 15
RANDOM_SEED        = 0
LEIDEN_RESOLUTIONS = [0.5, 0.8, 1.2]

CELL_TYPE_MARKERS = {
    "colonocyte_absorptive": ["EPCAM","SLC26A3","CA1","CA2","KRT20","CDX2","FABP1"],
    "colonocyte_best4":      ["BEST4","OTOP2","SPIB","CFTR","NOTCH2"],
    "goblet":                ["MUC2","TFF3","CLCA1","ZG16","FCGBP","SPDEF"],
    "stem_progenitor":       ["LGR5","OLFM4","ASCL2","SMOC2","SOX9","RGMB"],
    "paneth":                ["DEFA5","DEFA6","LYZ","PRSS2","ITLN2"],
    "enteroendocrine":       ["CHGA","CHGB","SYP","GCG","CCK","SST"],
    "fibroblast":            ["DCN","LUM","COL1A1","PDGFRA","FAP","THY1"],
    "inf_fibroblast":        ["IL13RA2","IL11","WNT5A","CXCL14","PDPN","TNFRSF11B"],
    "smooth_muscle":         ["ACTA2","MYH11","CNN1","DES","TAGLN"],
    "t_cell":                ["CD3D","CD3E","CD8A","CD4","TRAC","IL7R"],
    "b_cell":                ["CD79A","MS4A1","CD19","MZB1","JCHAIN","IGHG1"],
    "myeloid":               ["CD68","LYZ","S100A9","CD14","FCGR3A","IL1B"],
    "endothelial":           ["VWF","PECAM1","CDH5","CLDN5","ACKR1"],
    "mast_cell":             ["TPSAB1","TPSB2","CPA3","KIT"],
}

np.random.seed(RANDOM_SEED); random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)

print("Loading Script 01 output...")
adata = sc.read_h5ad(os.path.join(PROCESSED, "01_loaded.h5ad"))
print(f"  Loaded: {adata.n_obs:,} cells × {adata.n_vars} HVGs")

In [ ]:
# scVI training
scvi.settings.seed = RANDOM_SEED
scvi.model.SCVI.setup_anndata(adata, batch_key="sample")
model = scvi.model.SCVI(adata, **SCVI_PARAMS)
print(f"Training scVI on {adata.n_obs:,} cells × {adata.n_vars} HVGs...")
model.train(**SCVI_TRAIN_PARAMS, accelerator="auto")
final_loss = float(np.array(model.history["train_loss_epoch"].values[-1]).flat[0])
print(f"Training complete. Final loss: {final_loss:.2f}")
adata.obsm["X_scVI"] = model.get_latent_representation()
print(f"Latent shape: {adata.obsm['X_scVI'].shape}")

In [ ]:
# Neighbour graph + UMAP
print("Building neighbour graph and UMAP...")
sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=N_NEIGHBORS)
sc.tl.umap(adata)
print("UMAP complete")

In [ ]:
# Multi-resolution Leiden
res_results = {}
for res in LEIDEN_RESOLUTIONS:
    key = f"leiden_{res}"
    sc.tl.leiden(adata, resolution=res, key_added=key,
                 random_state=RANDOM_SEED, flavor="igraph",
                 n_iterations=2, directed=False)
    n = adata.obs[key].nunique()
    print(f"  Resolution {res}: {n} clusters")
    res_results[res] = {"key": key, "n_clusters": n}

# Score resolutions
rows = []
for res, info in res_results.items():
    key = info["key"]
    best_clusters = {}
    for ct, markers in CELL_TYPE_MARKERS.items():
        present = [m for m in markers if m in adata.var_names]
        if not present: continue
        cluster_means = {}
        for cl in adata.obs[key].unique():
            mask = adata.obs[key] == cl
            e = adata[mask, present].X
            if hasattr(e, "toarray"): e = e.toarray()
            cluster_means[cl] = float(e.mean())
        best_clusters[ct] = max(cluster_means, key=cluster_means.get)
    n_distinct = len(set(best_clusters.values()))
    n_resolved = len(best_clusters)
    rows.append({"resolution": res, "n_clusters": info["n_clusters"],
                 "n_celltypes_resolved": n_resolved,
                 "n_distinct_best_clusters": n_distinct,
                 "separation_score": n_distinct / max(n_resolved, 1)})

res_df = pd.DataFrame(rows).sort_values("separation_score", ascending=False)
print("\nResolution comparison:")
print(res_df.to_string(index=False))

recommended = float(res_df.iloc[0]["resolution"])
print(f"\nRecommended resolution: {recommended} ({int(res_df.iloc[0]['n_clusters'])} clusters)")

adata.obs["leiden"] = adata.obs[f"leiden_{recommended}"].copy()
adata.uns["recommended_leiden_resolution"] = recommended
adata.uns["pro_ibd_clusters"] = []

In [ ]:
out_02 = os.path.join(PROCESSED, "02_scvi.h5ad")
adata.write_h5ad(out_02)
res_df.to_csv(os.path.join(PROCESSED, "resolution_metrics.csv"), index=False)
cond_dist = adata.obs.groupby(["leiden", "condition"]).size().unstack(fill_value=0)
cond_dist.to_csv(os.path.join(PROCESSED, "cluster_condition_distribution.csv"))
print(f"Saved: {out_02}")
print(f"Clusters: {adata.obs['leiden'].nunique()}")
print("\nScript 02 complete ✓")

---
## Script 03 — Cell Type Annotation
Marker-based scoring against 14 intestinal cell types.  
**Output:** `processed/03_annotated.h5ad`

In [ ]:
CELL_TYPE_MARKERS_03 = {
    "colonocyte_absorptive": ["EPCAM","SLC26A3","CA1","CA2","KRT20","CDX2","FABP1"],
    "colonocyte_best4":      ["BEST4","OTOP2","SPIB","CFTR","NOTCH2"],
    "goblet":                ["MUC2","TFF3","CLCA1","ZG16","FCGBP","SPDEF"],
    "stem_progenitor":       ["LGR5","OLFM4","ASCL2","SMOC2","SOX9"],
    "paneth":                ["DEFA5","DEFA6","LYZ","PRSS2","ITLN2"],
    "enteroendocrine":       ["CHGA","CHGB","SYP","GCG","CCK"],
    "fibroblast":            ["DCN","LUM","COL1A1","PDGFRA","FAP"],
    "inf_fibroblast":        ["IL13RA2","IL11","WNT5A","CXCL14","PDPN"],
    "smooth_muscle":         ["ACTA2","MYH11","CNN1","DES","TAGLN"],
    "t_cell":                ["CD3D","CD3E","CD8A","IL7R","TRAC"],
    "b_cell":                ["CD79A","MS4A1","MZB1","XBP1","JCHAIN"],
    "myeloid":               ["CD68","LYZ","S100A9","CD14","IL1B"],
    "endothelial":           ["VWF","PECAM1","CDH5","CLDN5"],
    "mast_cell":             ["TPSAB1","TPSB2","CPA3","KIT"],
}
CONFIDENCE_THRESHOLD = 1.2

print("Loading Script 02 output...")
adata = sc.read_h5ad(os.path.join(PROCESSED, "02_scvi.h5ad"))
print(f"  Loaded: {adata.n_obs:,} cells, {adata.obs['leiden'].nunique()} clusters")

if "norm_log" not in adata.layers:
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

In [ ]:
# Score clusters against markers
clusters = sorted(adata.obs["leiden"].unique(), key=lambda x: int(x))
scores = {ct: [] for ct in CELL_TYPE_MARKERS_03}
for cl in clusters:
    mask = adata.obs["leiden"] == cl
    for ct, markers in CELL_TYPE_MARKERS_03.items():
        present = [m for m in markers if m in adata.var_names]
        if not present:
            scores[ct].append(0.0); continue
        e = adata[mask, present].X
        if hasattr(e, "toarray"): e = e.toarray()
        scores[ct].append(float(e.mean()))

score_df_03 = pd.DataFrame(scores, index=clusters)
score_df_03.index.name = "cluster"

# Assign annotations
ann_rows = []
for cluster in score_df_03.index:
    row = score_df_03.loc[cluster]
    best = row.idxmax(); best_score = row.max()
    sorted_s = row.sort_values(ascending=False)
    runner_up = sorted_s.iloc[1] if len(sorted_s) > 1 else 0.0
    if best_score == 0.0:
        label, conf = "unknown", "low"
    elif runner_up == 0.0 or best_score >= CONFIDENCE_THRESHOLD * runner_up:
        label, conf = best, "high"
    else:
        label, conf = best + "_mixed", "low"
    ann_rows.append({"cluster": cluster, "annotation": label, "confidence": conf,
                     "best_score": round(best_score, 4), "runner_up_score": round(runner_up, 4)})

ann_df = pd.DataFrame(ann_rows)
counts = adata.obs["leiden"].value_counts().rename("n_cells")
summary = ann_df.set_index("cluster").join(counts).sort_values("n_cells", ascending=False)

print("Cluster annotations:")
print("-" * 70)
for cl, row in summary.iterrows():
    print(f"  Cluster {cl:>3} | {row['annotation']:<30} | {row['confidence']:<4} | {int(row['n_cells']):>6} cells")

epi_n = ann_df["annotation"].str.contains("colonocyte|goblet|stem|paneth|enteroendocrine", na=False).sum()
print(f"\nEpithelial clusters: {epi_n}")

In [ ]:
adata.obs["cell_type"] = adata.obs["leiden"].map(
    dict(zip(ann_df["cluster"].astype(str), ann_df["annotation"])))
adata.obs["annotation_confidence"] = adata.obs["leiden"].map(
    dict(zip(ann_df["cluster"].astype(str), ann_df["confidence"])))

out_03 = os.path.join(PROCESSED, "03_annotated.h5ad")
adata.write_h5ad(out_03)
ann_df.to_csv(os.path.join(PROCESSED, "cluster_annotations.csv"), index=False)
score_df_03.to_csv(os.path.join(PROCESSED, "cluster_marker_scores.csv"))
print(f"Saved: {out_03}")
print("\nScript 03 complete ✓")

---
## Script 04 — CD Signature Scoring
Five Crohn's disease signatures scored per cell and per cluster. Primary: innate immune activation + Th1/Th17 axis.  
**Output:** `processed/04_scored.h5ad`

In [ ]:
CD_SIGNATURES = {
    "epithelial_barrier_dysfunction": [
        "CLDN1","CLDN2","CLDN4","OCLN","TJP1","MUC2","MUC5B",
        "SPDEF","KLF4","CDX2","FABP1","SLC26A3","DPP4","VIL1"
    ],
    "innate_immune_activation": [
        "S100A8","S100A9","IL1B","TNF","CXCL1","CXCL2","CXCL8",
        "IL6","IL18","NLRP3","CASP1","NOD2","RIPK2","PTGS2"
    ],
    "th1_th17_axis": [
        "IFNG","IL12A","IL12B","IL23A","IL17A","IL17F","IL22",
        "STAT1","STAT3","STAT4","TBX21","RORC","IRF1","CXCL10"
    ],
    "fibrosis_remodelling": [
        "COL1A1","COL1A2","COL3A1","FN1","ACTA2","TGFB1","TGFB2",
        "TIMP1","MMP1","MMP3","MMP9","CTGF","POSTN","PDGFRA"
    ],
    "inf_fibroblast_programme": [
        "IL13RA2","IL11","WNT5A","IL24","CXCL14","PDPN","TNFRSF11B",
        "MMP1","MMP3","CCL2","CCL7","CCL8","OSMR","FAP"
    ],
}

print("Loading Script 03 output...")
adata = sc.read_h5ad(os.path.join(PROCESSED, "03_annotated.h5ad"))
print(f"  Loaded: {adata.n_obs:,} cells × {adata.n_vars} genes")

if "norm_log" not in adata.layers:
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

In [ ]:
# Score signatures per cell and per cluster
clusters = sorted(adata.obs["leiden"].unique(), key=lambda x: int(x))
cluster_scores = {sig: [] for sig in CD_SIGNATURES}

print("Scoring CD signatures:")
for sig_name, genes in CD_SIGNATURES.items():
    present = [g for g in genes if g in adata.var_names]
    missing = [g for g in genes if g not in adata.var_names]
    msg = f"  {sig_name}: {len(present)}/{len(genes)} genes"
    if missing: msg += f" (missing: {missing[:3]}{'...' if len(missing)>3 else ''})"
    print(msg)
    if not present:
        adata.obs["score_" + sig_name] = 0.0
        cluster_scores[sig_name].extend([0.0] * len(clusters))
        continue
    e = adata[:, present].X
    if hasattr(e, "toarray"): e = e.toarray()
    cell_scores = np.array(e.mean(axis=1)).flatten()
    adata.obs["score_" + sig_name] = cell_scores
    for cl in clusters:
        mask = adata.obs["leiden"] == cl
        cluster_scores[sig_name].append(float(cell_scores[mask].mean()))

sig_score_df = pd.DataFrame(cluster_scores, index=clusters)
sig_score_df.index.name = "cluster"
sig_score_df["cd_primary_score"] = (
    sig_score_df.get("innate_immune_activation", 0) +
    sig_score_df.get("th1_th17_axis", 0))
sig_score_df["cd_composite_score"] = sig_score_df[
    [c for c in CD_SIGNATURES if c in sig_score_df.columns]].sum(axis=1)

top_clusters = sig_score_df.nlargest(3, "cd_primary_score")
print("\nTop 3 pro-CD clusters:")
for cl, row in top_clusters.iterrows():
    ct = adata.obs.loc[adata.obs["leiden"] == str(cl), "cell_type"].values
    ct = ct[0] if len(ct) > 0 else "unknown"
    print(f"  Cluster {cl} ({ct}): innate={row.get('innate_immune_activation',0):.4f}, "
          f"th1/17={row.get('th1_th17_axis',0):.4f}")

In [ ]:
# Scores by condition
print("Scores by condition:")
cond_rows = []
for cond in ["Involved", "Uninvolved"]:
    mask = adata.obs["condition"] == cond
    if not mask.any(): continue
    row = {"condition": cond, "n_cells": int(mask.sum())}
    for sig in CD_SIGNATURES:
        col = "score_" + sig
        row[sig] = round(float(adata.obs.loc[mask, col].mean()), 4) if col in adata.obs.columns else 0.0
    cond_rows.append(row)
cond_scores_df = pd.DataFrame(cond_rows)
print(cond_scores_df.round(4).to_string(index=False))

adata.uns["pro_ibd_clusters"] = top_clusters.index.tolist()

out_04 = os.path.join(PROCESSED, "04_scored.h5ad")
adata.write_h5ad(out_04)
sig_score_df.to_csv(os.path.join(PROCESSED, "signature_scores.csv"))
cond_scores_df.to_csv(os.path.join(PROCESSED, "signature_scores_by_condition.csv"), index=False)
print(f"\nSaved: {out_04}")
print("\nScript 04 complete ✓")

---
## Script 05 — Differential Expression
Three DE comparisons: (1) Involved vs Uninvolved all cells, (2) cluster-vs-rest, (3) epithelial Involved vs Uninvolved.  
**Output:** `processed/05_de.h5ad` + CSVs

In [ ]:
N_TOP_GENES_DE = 150

print("Loading Script 04 output...")
adata = sc.read_h5ad(os.path.join(PROCESSED, "04_scored.h5ad"))
n_clusters = adata.obs["leiden"].nunique()
print(f"  Loaded: {adata.n_obs:,} cells, {n_clusters} clusters")
print(f"  Involved: {(adata.obs['condition']=='Involved').sum():,} | "
      f"Uninvolved: {(adata.obs['condition']=='Uninvolved').sum():,}")

if "norm_log" not in adata.layers:
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

In [ ]:
# Primary DE: Involved vs Uninvolved (all cells)
print("Involved vs Uninvolved DE (all cells)...")
sc.tl.rank_genes_groups(adata, groupby="condition", groups=["Involved"],
                        reference="Uninvolved", method="wilcoxon",
                        use_raw=False, key_added="rank_genes_inv_vs_uni", pts=True)
result = adata.uns["rank_genes_inv_vs_uni"]
inv_vs_uni = pd.DataFrame({
    "gene":     result["names"]["Involved"],
    "score":    result["scores"]["Involved"],
    "pval_adj": result["pvals_adj"]["Involved"],
    "log2fc":   result["logfoldchanges"]["Involved"],
}).sort_values("score", ascending=False)
inv_vs_uni.to_csv(os.path.join(PROCESSED, "de_Involved_vs_Uninvolved.csv"), index=False)
print(f"  Top upregulated:   {inv_vs_uni.head(5)['gene'].tolist()}")
print(f"  Top downregulated: {inv_vs_uni.tail(5)['gene'].tolist()}")

In [ ]:
# Cluster-vs-rest DE
print("Cluster-vs-rest DE...")
sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon",
                        use_raw=False, key_added="rank_genes_groups", pts=True)
result2 = adata.uns["rank_genes_groups"]
rows_de = []
for cl in result2["names"].dtype.names:
    df_cl = pd.DataFrame({
        "cluster": cl,
        "gene":    result2["names"][cl],
        "score":   result2["scores"][cl],
        "pval_adj": result2["pvals_adj"][cl],
    })
    rows_de.extend([df_cl.nlargest(N_TOP_GENES_DE, "score").assign(direction="up"),
                    df_cl.nsmallest(N_TOP_GENES_DE, "score").assign(direction="down")])
top_genes_df = pd.concat(rows_de, ignore_index=True)
top_genes_df.to_csv(os.path.join(PROCESSED, "de_top_genes.csv"), index=False)
print(f"  Extracted {len(top_genes_df):,} gene-cluster pairs")

In [ ]:
# Epithelial-specific DE
if "cell_type" in adata.obs.columns:
    epi_mask = adata.obs["cell_type"].str.contains(
        "colonocyte|goblet|stem|paneth|enteroendocrine", na=False)
    adata_epi = adata[epi_mask].copy()
    print(f"Epithelial subset: {adata_epi.n_obs:,} cells")
    if adata_epi.n_obs > 100 and (adata_epi.obs["condition"] == "Involved").sum() > 50:
        sc.tl.rank_genes_groups(adata_epi, groupby="condition", groups=["Involved"],
                                reference="Uninvolved", method="wilcoxon",
                                use_raw=False, key_added="rank_epi_inv")
        epi_r = adata_epi.uns["rank_epi_inv"]
        epi_de = pd.DataFrame({
            "gene":     epi_r["names"]["Involved"],
            "score":    epi_r["scores"]["Involved"],
            "pval_adj": epi_r["pvals_adj"]["Involved"],
        }).sort_values("score", ascending=False)
        epi_de.to_csv(os.path.join(PROCESSED, "de_epithelial_Involved_vs_Uninvolved.csv"), index=False)
        print(f"  Epithelial top genes: {epi_de.head(5)['gene'].tolist()}")
    else:
        print("  Insufficient epithelial cells for focused DE.")

# Pro-IBD cluster focused DE
pro_clusters = [str(c) for c in list(adata.uns.get("pro_ibd_clusters", []))]
if pro_clusters:
    adata.obs["pro_ibd_group"] = adata.obs["leiden"].apply(
        lambda x: "pro_ibd" if str(x) in pro_clusters else "other")
    sc.tl.rank_genes_groups(adata, groupby="pro_ibd_group",
                            groups=["pro_ibd"], reference="other",
                            method="wilcoxon", use_raw=False,
                            key_added="rank_genes_proibd")
    pr = adata.uns["rank_genes_proibd"]
    pr_df = pd.DataFrame({
        "gene":     pr["names"]["pro_ibd"],
        "score":    pr["scores"]["pro_ibd"],
        "pval_adj": pr["pvals_adj"]["pro_ibd"],
    }).sort_values("score", ascending=False)
    pr_df.to_csv(os.path.join(PROCESSED, "de_proibd_vs_rest.csv"), index=False)
    print("  Pro-IBD cluster DE saved.")

out_05 = os.path.join(PROCESSED, "05_de.h5ad")
adata.write_h5ad(out_05)
print(f"\nSaved: {out_05}")
print("\nScript 05 complete ✓")

---
## Script 06 — LINCS L1000 Reversal Scoring
Enrichr queries: primary Involved vs Uninvolved, epithelial subset, all clusters. Compounds ranked by reversal score.  
**Output:** `processed/lincs_candidates.csv`

In [ ]:
import time, re as re_06
import gseapy as gp

N_TOP_GENES_LINCS = 150
TOP_PER_QUERY     = 15
ENRICHR_DELAY     = 1.0
ENRICHR_LIBRARIES = [
    "LINCS_L1000_Chem_Pert_up",
    "LINCS_L1000_Chem_Pert_down",
    "GO_Biological_Process_2023",
    "Reactome_2022",
    "KEGG_2021_Human",
]

def clean_compound_name(term):
    m = re_06.match(r'^LJP\d+\s+\S+\s+\S+?-(.+)-[\d.]+$', term.strip())
    if m: return m.group(1).strip()
    return term.split("_")[0].strip() if term else term.strip()

def run_enrichr(query_id, up_genes, down_genes):
    results = []
    for direction, genes, reversal_lib in [
        ("up",   up_genes,   "LINCS_L1000_Chem_Pert_down"),
        ("down", down_genes, "LINCS_L1000_Chem_Pert_up"),
    ]:
        if not genes: continue
        for lib in ENRICHR_LIBRARIES:
            try:
                enr = gp.enrichr(gene_list=genes, gene_sets=lib, outdir=None, verbose=False)
                df = enr.results.copy()
                if df.empty: continue
                df["query_id"]        = query_id
                df["query_direction"] = direction
                df["library"]         = lib
                if lib in ("LINCS_L1000_Chem_Pert_up", "LINCS_L1000_Chem_Pert_down"):
                    adj_p = df["Adjusted P-value"].clip(lower=1e-300)
                    sign  = 1.0 if lib == reversal_lib else -1.0
                    df["reversal_score"] = sign * (-np.log10(adj_p))
                    df["compound"] = df["Term"].apply(clean_compound_name)
                else:
                    df["reversal_score"] = 0.0
                    df["compound"] = df["Term"]
                results.append(df)
                time.sleep(ENRICHR_DELAY)
            except Exception as e:
                print(f"    WARNING: Enrichr failed [{query_id} {lib}]: {e}")
                time.sleep(ENRICHR_DELAY * 2)
    return pd.concat(results, ignore_index=True) if results else pd.DataFrame()

print("Loading DE gene lists...")
inv_vs_uni   = pd.read_csv(os.path.join(PROCESSED, "de_Involved_vs_Uninvolved.csv"))
top_genes_df = pd.read_csv(os.path.join(PROCESSED, "de_top_genes.csv"))
print(f"  Involved vs Uninvolved: {len(inv_vs_uni):,} genes")
print(f"  Cluster DE: {len(top_genes_df):,} gene-cluster pairs, {top_genes_df['cluster'].nunique()} clusters")

In [ ]:
all_results = []

# Primary: Involved vs Uninvolved
print("Primary query: Involved vs Uninvolved...")
inv_up   = inv_vs_uni[inv_vs_uni["score"] > 0].head(N_TOP_GENES_LINCS)["gene"].tolist()
inv_down = inv_vs_uni[inv_vs_uni["score"] < 0].tail(N_TOP_GENES_LINCS)["gene"].tolist()
prim_res = run_enrichr("Inv_vs_Uni", inv_up, inv_down)
if not prim_res.empty:
    all_results.append(prim_res)
    print(f"  {(prim_res['reversal_score'] > 0).sum()} reversal hits")

# Epithelial subset
epi_path = os.path.join(PROCESSED, "de_epithelial_Involved_vs_Uninvolved.csv")
if os.path.exists(epi_path):
    epi_de   = pd.read_csv(epi_path)
    epi_up   = epi_de[epi_de["score"] > 0].head(N_TOP_GENES_LINCS)["gene"].tolist()
    epi_down = epi_de[epi_de["score"] < 0].tail(N_TOP_GENES_LINCS)["gene"].tolist()
    print("Epithelial Inv vs Uni query...")
    epi_res = run_enrichr("Epithelial_Inv_vs_Uni", epi_up, epi_down)
    if not epi_res.empty:
        all_results.append(epi_res)
        print(f"  {(epi_res['reversal_score'] > 0).sum()} reversal hits")

# Cluster queries
n_clusters_lincs = top_genes_df["cluster"].nunique()
for i, cl in enumerate(top_genes_df["cluster"].unique()):
    print(f"  Cluster {cl} ({i+1}/{n_clusters_lincs})...", end=" ", flush=True)
    up   = top_genes_df.loc[(top_genes_df["cluster"] == cl) & (top_genes_df["direction"] == "up"), "gene"].tolist()
    down = top_genes_df.loc[(top_genes_df["cluster"] == cl) & (top_genes_df["direction"] == "down"), "gene"].tolist()
    res  = run_enrichr(f"cluster_{cl}", up, down)
    if not res.empty:
        all_results.append(res)
        print(f"{(res['reversal_score'] > 0).sum()} reversal hits")
    else:
        print("no results")

if not all_results:
    raise RuntimeError("No Enrichr results returned. Check network and gene lists.")

raw_results = pd.concat(all_results, ignore_index=True)
raw_results.to_csv(os.path.join(PROCESSED, "lincs_results_raw.csv"), index=False)
print(f"\nRaw results: {len(raw_results):,} rows")

In [ ]:
# Deduplicate and rank
lincs_mask = raw_results["library"].str.startswith("LINCS_L1000")
lincs_df   = raw_results[lincs_mask & (raw_results["reversal_score"] > 0)].copy()

top = lincs_df.sort_values("reversal_score", ascending=False).groupby("query_id").head(TOP_PER_QUERY)
candidates = top.groupby("compound").agg(
    max_reversal_score=("reversal_score", "max"),
    n_queries=("query_id", "nunique"),
    queries=("query_id", lambda x: ",".join(sorted(set(x.astype(str))))),
).reset_index().sort_values("max_reversal_score", ascending=False)

print(f"Unique compounds: {len(candidates)}")
print(candidates[["compound","max_reversal_score","n_queries"]].head(15).round(2).to_string(index=False))

candidates.to_csv(os.path.join(PROCESSED, "lincs_candidates.csv"), index=False)
print(f"\nSaved: lincs_candidates.csv")
print("\nScript 06 complete ✓")

---
## Script 07 — Novelty Assessment & Priority Scoring
PubMed novelty tiers (NOVEL_ALL / NOVEL_CD / KNOWN). Priority score = reversal × novelty weight × n_queries.  
**Output:** `processed/priority_candidates.csv`, `processed/patent_watch.csv`

In [ ]:
import requests

NCBI_ESEARCH = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
NCBI_DELAY   = 0.4
NCBI_EMAIL   = "glen.ritschel@ritschelresearch.com"
NOVELTY_WEIGHTS = {"NOVEL_ALL": 3.0, "NOVEL_CD": 1.5, "KNOWN": 1.0}
PATENT_WATCH_MIN_REVERSAL = 20.0
PATENT_WATCH_MIN_QUERIES  = 2

MOA_REFERENCE = {
    "pd-0325901": "MEK1/2 inhibitor",
    "selumetinib": "MEK1/2 inhibitor",
    "trametinib": "MEK1/2 inhibitor",
    "pd-184352": "MEK1/2 inhibitor",
    "azd-8330": "MEK1/2 inhibitor",
    "ldn-193189": "BMP receptor ALK2/ALK3 inhibitor",
    "wye-125132": "mTORC1/2 inhibitor",
    "rapamycin": "mTORC1 inhibitor",
    "everolimus": "mTORC1 inhibitor",
    "as-605240": "PI3K-gamma inhibitor",
    "fedratinib": "JAK2 inhibitor",
    "ruxolitinib": "JAK1/2 inhibitor",
    "tofacitinib": "JAK1/3 inhibitor",
    "baricitinib": "JAK1/2 inhibitor",
    "upadacitinib": "JAK1 inhibitor",
    "filgotinib": "JAK1 inhibitor",
    "radicicol": "HSP90 inhibitor",
    "geldanamycin": "HSP90 inhibitor",
    "withaferin-a": "NF-kB/HSP90 inhibitor",
    "celastrol": "NF-kB/HSP90 inhibitor",
    "cgp-60474": "CDK1/2 inhibitor",
    "palbociclib": "CDK4/6 inhibitor",
    "bi-2536": "PLK1 inhibitor",
    "xmd-1150": "ERK5 inhibitor",
    "wz-3105": "SRC/ABL inhibitor",
    "wz-4-145": "CDK8 inhibitor",
    "saracatinib": "SRC/ALK2 dual inhibitor",
    "alvocidib": "CDK1/2/4/6/9 inhibitor (flavopiridol)",
    "at-7519": "CDK1/2/4/6/9 inhibitor",
    "pf-431396": "FAK/PYK2 inhibitor",
    "azd-5438": "CDK inhibitor",
    "azd-7762": "CHK1/2 inhibitor",
    "ql-xii-47": "MELK/FLT3 inhibitor",
    "canertinib": "Pan-EGFR inhibitor",
    "gefitinib": "EGFR inhibitor",
    "infliximab": "anti-TNF monoclonal antibody",
    "adalimumab": "anti-TNF monoclonal antibody",
    "vedolizumab": "anti-integrin alpha4beta7",
    "ustekinumab": "IL-12/23 monoclonal antibody",
    "risankizumab": "IL-23 monoclonal antibody",
    "ozanimod": "S1P receptor modulator",
    "methotrexate": "Antifolate / immunosuppressant",
    "azathioprine": "Thiopurine / immunosuppressant",
    "mercaptopurine": "Thiopurine / immunosuppressant",
    "mitoxantrone": "Topoisomerase II inhibitor",
    "i-bet151": "BET bromodomain inhibitor",
    "i-bet": "BET bromodomain inhibitor",
    "nintedanib": "FGFR/PDGFR/VEGFR inhibitor",
    "pirfenidone": "TGF-beta / anti-fibrotic",
    "sb-431542": "TGF-beta receptor ALK5 inhibitor",
    "plx-4720": "BRAF V600E inhibitor",
    "chelerythrine chloride": "PKC inhibitor",
    "dovitinib": "VEGFR/FGFR/PDGFR inhibitor",
}

def pubmed_hit_count(query, retries=3):
    params = {"db": "pubmed", "term": query, "rettype": "count",
              "retmode": "json", "email": NCBI_EMAIL}
    for attempt in range(retries):
        try:
            resp = requests.get(NCBI_ESEARCH, params=params, timeout=10)
            resp.raise_for_status()
            count = int(resp.json()["esearchresult"]["count"])
            time.sleep(NCBI_DELAY)
            return count
        except Exception:
            if attempt < retries - 1: time.sleep(NCBI_DELAY * 3)
            else: return -1

def assess_novelty(compound_name):
    q = f'"{compound_name}"'
    hits_cd    = pubmed_hit_count(q + ' AND ("Crohn" OR "Crohn\'s disease" OR "inflammatory bowel")')
    hits_epi   = pubmed_hit_count(q + ' AND ("intestinal epithelium" OR "colonocyte" OR "ileal")')
    hits_immune = pubmed_hit_count(q + ' AND ("IL-17" OR "TNF" OR "IL-12" OR "IL-23" OR "colitis")')
    if hits_cd == 0 and hits_epi == 0 and hits_immune == 0:
        tier = "NOVEL_ALL"
    elif hits_cd == 0 and hits_epi == 0:
        tier = "NOVEL_CD"
    else:
        tier = "KNOWN"
    return {"compound": compound_name, "hits_cd": hits_cd,
            "hits_intestinal_epithelium": hits_epi,
            "hits_ibd_immune": hits_immune, "novelty_tier": tier}

print("Loading LINCS candidates...")
candidates = pd.read_csv(os.path.join(PROCESSED, "lincs_candidates.csv"))
print(f"  {len(candidates)} candidates to assess")

In [ ]:
# PubMed novelty assessment
novelty_rows = []
for i, row in candidates.iterrows():
    compound = row["compound"]
    print(f"  [{i+1}/{len(candidates)}] {compound}...", end=" ", flush=True)
    nov = assess_novelty(compound)
    novelty_rows.append(nov)
    print(f"{nov['novelty_tier']} "
          f"(CD:{nov['hits_cd']}, Epi:{nov['hits_intestinal_epithelium']}, "
          f"Immune:{nov['hits_ibd_immune']})")

novelty_df = pd.DataFrame(novelty_rows)
novelty_df.to_csv(os.path.join(PROCESSED, "novelty_raw.csv"), index=False)

In [ ]:
# Priority scoring
merged = candidates.merge(novelty_df, on="compound", how="left")
merged["novelty_tier"] = merged["novelty_tier"].fillna("KNOWN")
merged["moa"] = merged["compound"].apply(
    lambda x: MOA_REFERENCE.get(x.lower().strip(), "unknown"))
merged["priority_score"] = merged.apply(
    lambda r: round(r["max_reversal_score"] *
                    NOVELTY_WEIGHTS.get(r["novelty_tier"], 1.0) *
                    r["n_queries"], 1), axis=1)
merged = merged.sort_values("priority_score", ascending=False)

print("Novelty breakdown:")
print(merged["novelty_tier"].value_counts().to_string())

display_cols = ["compound","moa","novelty_tier","max_reversal_score","n_queries","priority_score"]
print("\nTop 20 priority candidates:")
print(merged[display_cols].head(20).round(2).to_string(index=False))

patent_watch = merged[
    (merged["novelty_tier"] == "NOVEL_ALL") &
    (merged["max_reversal_score"] >= PATENT_WATCH_MIN_REVERSAL) &
    (merged["n_queries"] >= PATENT_WATCH_MIN_QUERIES)
].copy()

print(f"\nPatent watch list: {len(patent_watch)} NOVEL_ALL compounds")
if not patent_watch.empty:
    print(patent_watch[display_cols].to_string(index=False))

merged.to_csv(os.path.join(PROCESSED, "priority_candidates.csv"), index=False)
patent_watch.to_csv(os.path.join(PROCESSED, "patent_watch.csv"), index=False)

print(f"\nSaved: priority_candidates.csv ({len(merged)} compounds)")
print(f"Saved: patent_watch.csv ({len(patent_watch)} compounds)")
print("\nScript 07 complete ✓")
print("=" * 60)
print("PIPELINE COMPLETE — review priority_candidates.csv")
print("=" * 60)